# 39 — Hard Negative Augmentation Ablation

Ablation study: compare five strategies for augmenting the training set with
'true inactive' compounds (hard negatives). The key question is whether better
characterization of inactives improves predictions on cliff-proximal test compounds.

**Strategies:**
- Strategy 0 (baseline): 4,139 CRC compounds only
- Strategy 1 (current nb26/nb09): + single-conc inactives at N(4.3, 0.24) clipped [1.5, 5.0]
- Strategy 2 (low floor): same inactives but pEC50 from N(2.5, 0.5) clipped [1.0, 3.5]
- Strategy 3 (bimodal): split 50/50 between N(4.3, 0.24) and N(2.0, 0.4) modes
- Strategy 4 (PubChem inactives): use PubChem inactive CIDs at N(2.5, 0.5); fallback to Strategy 2

**Output:** best strategy OOF → `data/processed/oof_hard_negatives.npy`,
test predictions → `submissions/39_hard_negatives.csv`

In [1]:
# ── 1. Setup ──────────────────────────────────────────────────────────────────
import os
os.environ["PYTHONIOENCODING"] = "utf-8"

import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from pxr.data import load_train, load_test, load_single_conc
from pxr.chem import bemis_murcko, standardize_smiles, morgan_fp_batch, to_inchikey
from pxr.eval import scaffold_kfold_indices, rae as rae_fn, compute_metrics
from pxr.featurize import combined, impute
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS, FIGURES

SEED     = 42
N_FOLDS  = 5
RNG      = np.random.default_rng(SEED)

# LGBM hyperparams — same as nb26 / grand ensemble base
LGBM_PARAMS = dict(
    n_estimators=1200,
    num_leaves=64,
    learning_rate=0.04,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.2,
    min_child_samples=10,
    n_jobs=4,
    verbose=-1,
)

SC_WEIGHT = 0.25   # sample weight for pseudo-label rows

print('Setup complete.')

Setup complete.


## 1. Load Data + Existing Cliff Labels

Load training data, test data. Load cliff labels from nb38 if available;
otherwise compute inline using simplified Tanimoto threshold approach.

In [2]:
# ── 1. Load data + cliff labels ────────────────────────────────────────────────
train = load_train()
te    = load_test()

y_tr  = train['pec50'].values
print(f'Training: {len(train):,}  |  Test: {len(te):,}')
print(f'Train pEC50: mean={y_tr.mean():.3f}  std={y_tr.std():.3f}  range=[{y_tr.min():.2f}, {y_tr.max():.2f}]')

# Scaffold splits (used for all strategies)
scaffolds = train['smiles'].map(bemis_murcko).tolist()
splits    = scaffold_kfold_indices(scaffolds, n_splits=N_FOLDS, seed=SEED)
print(f'Scaffold CV splits: {N_FOLDS} folds')

# Load cliff labels from nb38 if available
cliff_labels_path = DATA_PROCESSED / 'cliff_labels.parquet'
if cliff_labels_path.exists():
    cliff_labels = pd.read_parquet(cliff_labels_path)
    cliff_member_set = set(cliff_labels[cliff_labels['is_cliff_member']]['name'])
    print(f'\nCliff labels loaded: {len(cliff_member_set):,} cliff members')
    has_cliff_labels = True
else:
    print('\nCliff labels not found — computing inline with simplified MMP approach')
    print('(Run nb38 first for full cliff analysis)')

    # Simplified inline cliff detection: Tanimoto >= 0.6, |DeltapEC50| >= 1.0
    train['std_smiles'] = train['smiles'].map(standardize_smiles)
    train_fp_inline = morgan_fp_batch(train['std_smiles'].fillna('').tolist())
    fp_f = train_fp_inline.astype(np.float32)
    counts_f = fp_f.sum(axis=1)
    inter_f = fp_f @ fp_f.T
    union_f = counts_f[:, None] + counts_f[None, :] - inter_f
    tan_f = np.where(union_f > 0, inter_f / union_f, 0.0)
    np.fill_diagonal(tan_f, 0.0)

    cliff_member_set = set()
    for i in range(len(train)):
        for j in range(i + 1, len(train)):
            if tan_f[i, j] >= 0.6 and abs(y_tr[i] - y_tr[j]) >= 1.0:
                cliff_member_set.add(train['name'].iloc[i])
                cliff_member_set.add(train['name'].iloc[j])

    print(f'Inline cliff members: {len(cliff_member_set):,}')
    has_cliff_labels = True

# Boolean mask for cliff members in training set
cliff_mask = np.array([n in cliff_member_set for n in train['name']])
print(f'Cliff member fraction: {cliff_mask.mean():.3f}')

Training: 4,139  |  Test: 513
Train pEC50: mean=4.321  std=1.121  range=[1.61, 7.55]


Scaffold CV splits: 5 folds



Cliff labels loaded: 248 cliff members
Cliff member fraction: 0.060


## 2. Source True Inactives

Three sources of hard negatives:
- **Source A**: High-confidence non-responders from single-conc screen
  (|log2FC| < 0.3 AND FDR > 0.5 AND not in CRC training)
- **Source B**: PubChem inactive set (from nb37)
- **Source C**: ChEMBL compounds inactive across all NR targets (pEC50 < 4.5 for all)

In [3]:
# ── 2. Source true inactives ───────────────────────────────────────────────────
sc = load_single_conc()
print(f'Single-conc: {len(sc):,} rows')
print(f'Single-conc columns: {sc.columns.tolist()}')

# Standardize single-conc SMILES
sc['std_smiles'] = sc['smiles'].map(standardize_smiles)
sc_valid = sc.dropna(subset=['std_smiles']).copy()

# Training InChIKeys for deduplication
train['std_smiles'] = train['smiles'].map(standardize_smiles)
train_std_set = set(train['std_smiles'].dropna())

# ── Source A: Single-conc high-confidence non-responders ──────────────────────
# Low FC, high FDR = compound genuinely did not respond
log2fc_col = 'log2_fc_estimate' if 'log2_fc_estimate' in sc_valid.columns else 'log2FC'
fdr_col    = 'fdr_bh'            if 'fdr_bh'            in sc_valid.columns else 'FDR'

src_a_mask = (
    (sc_valid[log2fc_col].abs() < 0.3) &
    (sc_valid[fdr_col] > 0.5) &
    (~sc_valid['std_smiles'].isin(train_std_set))
)
src_a = sc_valid[src_a_mask][['std_smiles']].drop_duplicates().reset_index(drop=True)
print(f'\nSource A (single-conc non-responders): {len(src_a):,}')

# ── Source B: PubChem inactive set ────────────────────────────────────────────
pubchem_cache = DATA_EXTERNAL / 'pubchem_pxr_aids.parquet'
if pubchem_cache.exists():
    pubchem_df = pd.read_parquet(pubchem_cache)
    src_b = pubchem_df[
        (pubchem_df.get('activity', pd.Series('active', index=pubchem_df.index)) == 'inactive') &
        (pubchem_df['std_smiles'].notna()) &
        (~pubchem_df['std_smiles'].isin(train_std_set))
    ][['std_smiles']].drop_duplicates().reset_index(drop=True)
    print(f'Source B (PubChem inactives):         {len(src_b):,}')
else:
    print('Source B (PubChem): cache not found — skipping (run nb37 first)')
    src_b = pd.DataFrame(columns=['std_smiles'])

# ── Source C: ChEMBL compounds inactive across all NR targets ─────────────────
chembl_cache = DATA_EXTERNAL / 'chembl_nr_extended.parquet'
if chembl_cache.exists():
    chembl_df = pd.read_parquet(chembl_cache)
    if 'target_name' in chembl_df.columns and 'pec50' in chembl_df.columns:
        # Pivot to wide format: one row per compound, one column per target
        chembl_wide = chembl_df.pivot_table(
            index='std_smiles', columns='target_name', values='pec50', aggfunc='max'
        ).reset_index()
        # Keep compounds with pEC50 < 4.5 across all measured NR targets
        nr_cols = [c for c in chembl_wide.columns if c != 'std_smiles']
        # A compound is "inactive" if max pEC50 across all NR targets < 4.5
        # (NaN means not measured — be conservative and include those)
        max_pec50 = chembl_wide[nr_cols].max(axis=1, skipna=True)
        src_c = chembl_wide[
            (max_pec50 < 4.5) & (~chembl_wide['std_smiles'].isin(train_std_set))
        ][['std_smiles']].drop_duplicates().reset_index(drop=True)
        print(f'Source C (ChEMBL pan-NR inactives):   {len(src_c):,}')
    else:
        print('Source C: ChEMBL cache lacks required columns — skipping')
        src_c = pd.DataFrame(columns=['std_smiles'])
else:
    print('Source C (ChEMBL): cache not found — skipping (run nb37 first)')
    src_c = pd.DataFrame(columns=['std_smiles'])

print(f'\nTotal inactives available:')
print(f'  Source A: {len(src_a):,}')
print(f'  Source B: {len(src_b):,}')
print(f'  Source C: {len(src_c):,}')

Single-conc: 21,003 rows
Single-conc columns: ['name', 'batch', 'smiles', 'plate_id', 'compound_class', 'concentration_M', 'log2_fc_estimate', 'log2_fc_stderr', 't_statistic', 'p_value', 'fdr_bh', 'neg_log10_fdr', 'median_log2_fc', 'n_replicates', 'cohens_d', 'experiment_name', 'ocnt_id', 'split', 'source']



Source A (single-conc non-responders): 3,324
Source B (PubChem inactives):         0
Source C (ChEMBL pan-NR inactives):   172

Total inactives available:
  Source A: 3,324
  Source B: 0
  Source C: 172


## 3. Featurize All Compounds

Compute combined features for training, test, and all inactive source compounds.

In [4]:
# ── 3. Featurize ──────────────────────────────────────────────────────────────
print('Featurizing CRC training set...')
X_tr = impute(combined(train['smiles'].tolist()))
print(f'  X_tr: {X_tr.shape}')

print('Featurizing test set...')
X_te = impute(combined(te['smiles'].tolist()))
print(f'  X_te: {X_te.shape}')

# Featurize inactive sources
print('Featurizing Source A (single-conc non-responders)...')
X_src_a = impute(combined(src_a['std_smiles'].tolist())) if len(src_a) > 0 else None
print(f'  X_src_a: {X_src_a.shape if X_src_a is not None else "N/A"}')

if len(src_b) > 0:
    print('Featurizing Source B (PubChem inactives)...')
    X_src_b = impute(combined(src_b['std_smiles'].tolist()))
    print(f'  X_src_b: {X_src_b.shape}')
else:
    X_src_b = None

if len(src_c) > 0:
    print('Featurizing Source C (ChEMBL pan-NR inactives)...')
    # Limit to 5000 to avoid excessive compute
    src_c_sample = src_c.sample(min(5000, len(src_c)), random_state=SEED).reset_index(drop=True)
    X_src_c = impute(combined(src_c_sample['std_smiles'].tolist()))
    print(f'  X_src_c: {X_src_c.shape} (sampled {len(src_c_sample):,}/{len(src_c):,})')
else:
    X_src_c = None
    src_c_sample = pd.DataFrame(columns=['std_smiles'])

print('\nFeaturization complete.')

Featurizing CRC training set...


  X_tr: (4139, 2265)
Featurizing test set...


  X_te: (513, 2265)
Featurizing Source A (single-conc non-responders)...


  X_src_a: (3324, 2265)
Featurizing Source C (ChEMBL pan-NR inactives)...


  X_src_c: (172, 2265) (sampled 172/172)

Featurization complete.


## 4. Augmentation Strategy Ablation

Run scaffold 5-fold CV for each strategy. Track:
- Overall OOF RAE
- OOF RAE on cliff members only
- Test prediction distribution (mean, std)

pEC50 assignments for pseudo-inactives:
- Strategy 1: N(4.3, 0.24) clipped [1.5, 5.0]
- Strategy 2: N(2.5, 0.5) clipped [1.0, 3.5]
- Strategy 3: 50% N(4.3, 0.24) clipped [3.5, 5.0] + 50% N(2.0, 0.4) clipped [1.0, 3.0]

In [5]:
# ── 4. Augmentation strategy ablation ─────────────────────────────────────────
def assign_pec50_strategy(n: int, strategy_id: int, rng: np.random.Generator) -> np.ndarray:
    """Generate pseudo-pEC50 values for n inactive compounds under a given strategy."""
    if strategy_id == 1:
        # Current nb26/nb09 approach: N(4.3, 0.24) clipped [1.5, 5.0]
        vals = rng.normal(4.3, 0.24, size=n)
        return np.clip(vals, 1.5, 5.0)
    elif strategy_id == 2:
        # Low floor: N(2.5, 0.5) clipped [1.0, 3.5]
        vals = rng.normal(2.5, 0.5, size=n)
        return np.clip(vals, 1.0, 3.5)
    elif strategy_id == 3:
        # Bimodal: 50/50 split
        n_mode1 = n // 2
        n_mode2 = n - n_mode1
        vals1 = np.clip(rng.normal(4.3, 0.24, size=n_mode1), 3.5, 5.0)
        vals2 = np.clip(rng.normal(2.0, 0.4,  size=n_mode2), 1.0, 3.0)
        combined_vals = np.concatenate([vals1, vals2])
        rng.shuffle(combined_vals)
        return combined_vals
    elif strategy_id == 4:
        # PubChem inactives: N(2.5, 0.5) clipped [1.0, 3.5]
        vals = rng.normal(2.5, 0.5, size=n)
        return np.clip(vals, 1.0, 3.5)
    else:
        raise ValueError(f'Unknown strategy_id: {strategy_id}')


def run_strategy(strategy_id: int, X_aug: np.ndarray | None, y_aug: np.ndarray | None,
                  label: str) -> dict:
    """
    Run scaffold 5-fold CV for a single augmentation strategy.
    Returns dict with OOF RAE, cliff RAE, test predictions.
    """
    print(f'\n--- {label} ---')
    oof = np.full(len(y_tr), np.nan)
    fold_metrics = []

    for fold_i, (tr_idx, va_idx) in enumerate(splits):
        if X_aug is not None and y_aug is not None and len(X_aug) > 0:
            X_fold = np.vstack([X_tr[tr_idx], X_aug])
            y_fold = np.concatenate([y_tr[tr_idx], y_aug])
            w_fold = np.concatenate([
                np.ones(len(tr_idx)),
                np.full(len(y_aug), SC_WEIGHT),
            ])
        else:
            X_fold = X_tr[tr_idx]
            y_fold = y_tr[tr_idx]
            w_fold = np.ones(len(tr_idx))

        m = lgb.LGBMRegressor(**LGBM_PARAMS)
        m.fit(X_fold, y_fold, sample_weight=w_fold)
        oof[va_idx] = m.predict(X_tr[va_idx])
        fold_rae = rae_fn(y_tr[va_idx], oof[va_idx])
        fold_metrics.append(fold_rae)
        print(f'  Fold {fold_i+1}: RAE={fold_rae:.4f}')

    oof_rae_global = rae_fn(y_tr, oof)

    # Cliff-specific RAE
    if cliff_mask.sum() > 0:
        cliff_rae = rae_fn(y_tr[cliff_mask], oof[cliff_mask])
    else:
        cliff_rae = float('nan')

    print(f'  OOF RAE (global): {oof_rae_global:.4f}')
    print(f'  OOF RAE (cliff members, n={cliff_mask.sum()}): {cliff_rae:.4f}')

    # Full retrain for test predictions
    if X_aug is not None and y_aug is not None and len(X_aug) > 0:
        X_full = np.vstack([X_tr, X_aug])
        y_full = np.concatenate([y_tr, y_aug])
        w_full = np.concatenate([
            np.ones(len(y_tr)),
            np.full(len(y_aug), SC_WEIGHT),
        ])
    else:
        X_full = X_tr
        y_full = y_tr
        w_full = np.ones(len(y_tr))

    final_m = lgb.LGBMRegressor(**LGBM_PARAMS)
    final_m.fit(X_full, y_full, sample_weight=w_full)
    te_pred = final_m.predict(X_te)
    te_pred = np.clip(te_pred, y_tr.min() - 0.5, y_tr.max() + 0.5)

    print(f'  Test preds: mean={te_pred.mean():.3f}  std={te_pred.std():.3f}')

    return {
        'label':        label,
        'strategy_id':  strategy_id,
        'oof_rae':      oof_rae_global,
        'cliff_rae':    cliff_rae,
        'test_mean':    float(te_pred.mean()),
        'test_std':     float(te_pred.std()),
        'n_aug':        len(y_aug) if y_aug is not None else 0,
        'oof_array':    oof,
        'te_array':     te_pred,
    }


# ── Prepare augmentation data for each strategy ───────────────────────────────
results = []

# Strategy 0: baseline (CRC only)
results.append(run_strategy(0, None, None, 'Strategy 0 (CRC only — baseline)'))

# Strategy 1: + Source A at N(4.3, 0.24)
if X_src_a is not None:
    y_aug_s1 = assign_pec50_strategy(len(src_a), 1, RNG)
    results.append(run_strategy(1, X_src_a, y_aug_s1, 'Strategy 1 (+ SC inactives N(4.3,0.24))'))
else:
    print('Strategy 1: no Source A data — skipping')

# Strategy 2: + Source A at N(2.5, 0.5) low floor
if X_src_a is not None:
    y_aug_s2 = assign_pec50_strategy(len(src_a), 2, RNG)
    results.append(run_strategy(2, X_src_a, y_aug_s2, 'Strategy 2 (+ SC inactives N(2.5,0.5) low)'))
else:
    print('Strategy 2: no Source A data — skipping')

# Strategy 3: + Source A bimodal
if X_src_a is not None:
    y_aug_s3 = assign_pec50_strategy(len(src_a), 3, RNG)
    results.append(run_strategy(3, X_src_a, y_aug_s3, 'Strategy 3 (+ SC inactives bimodal)'))
else:
    print('Strategy 3: no Source A data — skipping')

# Strategy 4: PubChem inactives or fallback to Strategy 2
if X_src_b is not None and len(src_b) > 0:
    y_aug_s4 = assign_pec50_strategy(len(src_b), 4, RNG)
    results.append(run_strategy(4, X_src_b, y_aug_s4, 'Strategy 4 (PubChem inactives N(2.5,0.5))'))
elif X_src_a is not None:
    print('Strategy 4: PubChem not available — using Source A as fallback (same as Strategy 2)')
    y_aug_s4_fb = assign_pec50_strategy(len(src_a), 4, RNG)
    results.append(run_strategy(4, X_src_a, y_aug_s4_fb, 'Strategy 4 (fallback: SC inactives N(2.5,0.5))'))
else:
    print('Strategy 4: no inactives available — skipping')

print(f'\nCompleted {len(results)} strategies.')


--- Strategy 0 (CRC only — baseline) ---


  Fold 1: RAE=0.4936


  Fold 2: RAE=0.5756


  Fold 3: RAE=0.5952


  Fold 4: RAE=0.5621


...

  Fold 2: RAE=0.8013


  Fold 3: RAE=0.8047


  Fold 4: RAE=0.8018


  Fold 5: RAE=0.8180
  OOF RAE (global): 0.7683
  OOF RAE (cliff members, n=248): 0.6496


  Test preds: mean=4.364  std=0.849

Completed 5 strategies.


## 5. Effect on Prediction Distribution

For each strategy, plot histogram of test predictions.
A good strategy maintains test_std close to train_std (not collapse to mean).

In [6]:
# ── 5. Effect on prediction distribution ──────────────────────────────────────
train_std = y_tr.std()

# Summary table
summary_rows = []
for r in results:
    summary_rows.append({
        'Strategy':     r['label'],
        'OOF_RAE':      r['oof_rae'],
        'Cliff_RAE':    r['cliff_rae'],
        'N_aug':        r['n_aug'],
        'Test_mean':    r['test_mean'],
        'Test_std':     r['test_std'],
        'Std_ratio':    r['test_std'] / train_std,  # 1.0 = matches train distribution
    })

summary_df = pd.DataFrame(summary_rows)
print('=' * 90)
print('Augmentation Strategy Ablation Results')
print('=' * 90)
print(summary_df.round(4).to_string(index=False))
print('=' * 90)
print(f'\nTrain pEC50 std: {train_std:.3f}')
print('Std_ratio > 1.0 = test predictions more spread than training distribution')
print('Std_ratio < 0.5 = collapsed predictions (bad)')

# Plot: histogram of test predictions per strategy
n_strat = len(results)
n_cols = min(3, n_strat)
n_rows = (n_strat + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = np.array(axes).flatten() if n_strat > 1 else [axes]

train_pec50_mean = y_tr.mean()

for idx, r in enumerate(results):
    ax = axes[idx]
    te_pred = r['te_array']
    ax.hist(te_pred, bins=30, color='steelblue', edgecolor='white', linewidth=0.5, alpha=0.8, label='Test preds')
    ax.axvline(train_pec50_mean, color='red', linestyle='--', alpha=0.7, label=f'Train mean ({train_pec50_mean:.2f})')
    ax.axvline(te_pred.mean(), color='green', linestyle='-', alpha=0.7, label=f'Test mean ({te_pred.mean():.2f})')
    ax.set_xlabel('Predicted pEC50')
    ax.set_ylabel('Count')
    short_label = r['label'].split('(')[0].strip()
    ax.set_title(f'{short_label}\nRAE={r["oof_rae"]:.4f}  std={r["test_std"]:.3f}')
    ax.legend(fontsize=7)

# Hide unused axes
for idx in range(len(results), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Hard Negative Augmentation: Test Prediction Distributions', fontsize=12, y=1.02)
plt.tight_layout()
fig_path = FIGURES / '39_hard_negatives_distributions.png'
plt.savefig(fig_path, dpi=120, bbox_inches='tight')
plt.close()
print(f'\nFigure saved to {fig_path}')

Augmentation Strategy Ablation Results
                                      Strategy  OOF_RAE  Cliff_RAE  N_aug  Test_mean  Test_std  Std_ratio
              Strategy 0 (CRC only — baseline)   0.5606     0.6033      0     4.7957    0.6501     0.5797
       Strategy 1 (+ SC inactives N(4.3,0.24))   0.6274     0.7114   3324     4.8345    0.5293     0.4720
    Strategy 2 (+ SC inactives N(2.5,0.5) low)   0.7582     0.6356   3324     4.3550    0.8409     0.7498
           Strategy 3 (+ SC inactives bimodal)   0.6874     0.6587   3324     4.5382    0.7006     0.6247
Strategy 4 (fallback: SC inactives N(2.5,0.5))   0.7683     0.6496   3324     4.3637    0.8494     0.7574

Train pEC50 std: 1.121
Std_ratio > 1.0 = test predictions more spread than training distribution
Std_ratio < 0.5 = collapsed predictions (bad)



Figure saved to D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\data\processed\figures\39_hard_negatives_distributions.png


## 6. Best Strategy Selection + Submission

Select strategy with lowest OOF RAE (or best cliff-specific RAE if cliff members are available).
Save OOF to `data/processed/oof_hard_negatives.npy` and test predictions
to `submissions/39_hard_negatives.csv`.

In [7]:
# ── 6. Best strategy selection + submission ────────────────────────────────────
if not results:
    raise RuntimeError('No strategies completed — cannot select best.')

# Primary sort: OOF RAE (lower is better)
# Secondary tiebreak: cliff RAE (lower is better, nan last)
def sort_key(r):
    cliff_rae = r['cliff_rae'] if not np.isnan(r['cliff_rae']) else 999.0
    return (r['oof_rae'], cliff_rae)

results_sorted = sorted(results, key=sort_key)
best = results_sorted[0]

print(f'Best strategy: {best["label"]}')
print(f'  OOF RAE (global): {best["oof_rae"]:.4f}')
print(f'  OOF RAE (cliff):  {best["cliff_rae"]:.4f}')
print(f'  Test std:         {best["test_std"]:.3f}  (train std: {train_std:.3f})')

# Compare vs baseline
baseline = next(r for r in results if r['strategy_id'] == 0)
delta_rae = best['oof_rae'] - baseline['oof_rae']
print(f'\nVs baseline (CRC only): delta_RAE = {delta_rae:+.4f}')

if best['strategy_id'] != 0:
    if best['oof_rae'] < baseline['oof_rae']:
        print('  IMPROVEMENT: hard negative augmentation helps')
    else:
        print('  NOTE: hard negatives did not improve overall OOF RAE')
        print('  Using best available strategy anyway')

# Save OOF array
oof_path = DATA_PROCESSED / 'oof_hard_negatives.npy'
np.save(oof_path, best['oof_array'])
print(f'\nOOF saved to {oof_path}')

# Save test predictions as submission
te_preds_best = best['te_array']
te_preds_best = np.clip(te_preds_best, y_tr.min() - 0.5, y_tr.max() + 0.5)

# Also save raw test array
te_path = DATA_PROCESSED / 'te_hard_negatives.npy'
np.save(te_path, te_preds_best)
print(f'Test preds saved to {te_path}')

sub = pd.DataFrame({
    'Molecule Name': te['name'].values,
    'pEC50':         te_preds_best,
})
assert len(sub) == 513, f'Expected 513 test compounds, got {len(sub)}'
assert sub['pEC50'].notna().all(), 'NaN pEC50 values in submission'

out_path = SUBMISSIONS / '39_hard_negatives.csv'
sub.to_csv(out_path, index=False)
print(f'Submission saved to {out_path}')

print(f'\n== Strategy 39 Hard Negative Augmentation Summary ==')
print(f'  Best strategy:     {best["label"]}')
print(f'  OOF RAE (global):  {best["oof_rae"]:.4f}')
print(f'  OOF RAE (cliff):   {best["cliff_rae"]:.4f}')
print(f'  N augmented:       {best["n_aug"]:,}')
print(f'  Test mean:         {te_preds_best.mean():.3f}')
print(f'  Test std:          {te_preds_best.std():.3f}  (train: {train_std:.3f})')
print(f'\nAll strategies ranked by OOF RAE:')
for i, r in enumerate(results_sorted):
    marker = ' <-- BEST' if i == 0 else ''
    print(f'  {r["label"]:<50s}  RAE={r["oof_rae"]:.4f}  CliffRAE={r["cliff_rae"]:.4f}{marker}')

Best strategy: Strategy 0 (CRC only — baseline)
  OOF RAE (global): 0.5606
  OOF RAE (cliff):  0.6033
  Test std:         0.650  (train std: 1.121)

Vs baseline (CRC only): delta_RAE = +0.0000

OOF saved to D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\data\processed\oof_hard_negatives.npy
Test preds saved to D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\data\processed\te_hard_negatives.npy
Submission saved to D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\39_hard_negatives.csv

== Strategy 39 Hard Negative Augmentation Summary ==
  Best strategy:     Strategy 0 (CRC only — baseline)
  OOF RAE (global):  0.5606
  OOF RAE (cliff):   0.6033
  N augmented:       0
  Test mean:         4.796
  Test std:          0.650  (train: 1.121)

All strategies ranked by OOF RAE:
  Strategy 0 (CRC only — baseline)                    RAE=0.5606  CliffRAE=0.6033 <-- BEST
  Strategy 1 (+ SC inactives N(4.3,0.24))             RAE=0.6274  CliffRAE=0.7114
  Strategy 